# A Multi-Index RAG Pipeline (Lesson 38)

Follow-along notebook for *A Multi-Index RAG pipeline* in Building with the Claude API.

The lesson's `Retriever` code is a sketch (its loop iterates over an empty list), and it assumes `VectorIndex` and `BM25Index` with matching `add_document` / `search` APIs. This notebook provides working versions of all of them:

- a `SearchIndex` protocol defining the shared interface
- `VectorIndex` and `BM25Index` that both take **documents** and **query text** (the vector index embeds internally)
- a complete `Retriever` with reciprocal rank fusion (RRF)
- an extra index added later to show the extensibility point

**Needs:** `VOYAGE_API_KEY` (and `ANTHROPIC_API_KEY` for the optional last section). **Run order:** Run All after a kernel restart.

## 1. Setup and the sample report

In [1]:
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Protocol, Tuple

import voyageai
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
voyage_client = voyageai.Client(max_retries=3)
anthropic_client = Anthropic()
EMBED_MODEL = "voyage-3-large"
MODEL = "claude-haiku-4-5"

REPORT = r'''# Annual Interdisciplinary Research Review 2023

## Executive Summary
This review summarizes the year's work across five research and operations divisions. Highlights include a successful Phase II clinical trial, a major platform release from software engineering, improved financial performance, and the resolution of a significant security incident. Each section below covers goals, results, and next steps.

## Table of Contents
1. Methodology
2. Section 1: Medical Research
3. Section 2: Software Engineering
4. Section 3: Financial Analysis
5. Section 4: Cybersecurity
6. Section 5: Legal and Compliance
7. Conclusion

## Methodology
Each division reported quarterly metrics using a shared template. Teams documented their processes, tools, and outcomes, and an independent review panel validated the figures. Engineering-heavy divisions also submitted system logs and release notes, which were cross-checked against project tracking data.

## Section 1: Medical Research
The medical research team completed Phase II trials for compound NR-204, a treatment for chronic inflammation. Enrollment paused briefly after a stomach bug spread through one trial site; the bug was traced to a catering supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers, and Phase III planning has begun.

## Section 2: Software Engineering
The software engineering department shipped four major releases of the analytics platform. Engineers fixed 1,247 bugs, a 22 percent increase over the prior year, and reduced average resolution time from nine days to four. The team migrated core services to a new container platform and raised automated test coverage to 81 percent.

## Section 3: Financial Analysis
Total revenue reached 96 million dollars, up 18 percent. Operating margin improved to 14 percent on lower infrastructure costs. Research spending totaled 21 million dollars, most of it in medical research. Cash reserves at year end were 38 million dollars.

## Section 4: Cybersecurity
The security team handled 37 incidents during the year, most of them low severity. The most significant, incident INC-2023-Q4-011, involved unauthorized access to a staging database through a misconfigured API gateway. The team contained it within six hours, rotated all affected credentials, and found no evidence that customer data left the environment. A post-incident review led to mandatory configuration audits for all gateways.

## Section 5: Legal and Compliance
The legal team renewed data processing agreements with all major vendors and completed an external privacy audit with no material findings. Following incident INC-2023-Q4-011, counsel confirmed that no regulatory notification was required.

## Conclusion
The organization ends the year in a strong position. Priorities for next year include Phase III trials, expanding the analytics platform into healthcare, and continued investment in security controls.
'''

if not Path("report.md").exists() or "INC-2023-Q4-011" not in Path("report.md").read_text(encoding="utf-8"):
    Path("report.md").write_text(REPORT, encoding="utf-8")


def chunk_by_section(document_text):
    return [c.strip() for c in re.split(r"\n(?=## )", document_text) if c.strip()]


chunks = chunk_by_section(Path("report.md").read_text(encoding="utf-8"))
for i, c in enumerate(chunks):
    print(i, c.splitlines()[0])

c:\Users\bspedden\OneDrive - The Select Group LLC\Documents\work\Building-With-The-Claude-API\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0 # Annual Interdisciplinary Research Review 2023
1 ## Executive Summary
2 ## Table of Contents
3 ## Methodology
4 ## Section 1: Medical Research
5 ## Section 2: Software Engineering
6 ## Section 3: Financial Analysis
7 ## Section 4: Cybersecurity
8 ## Section 5: Legal and Compliance
9 ## Conclusion


## 2. Reciprocal rank fusion by hand

Before any classes, the formula on the lesson's own example. Each index ranks documents; RRF ignores the raw scores (which aren't comparable: cosine distance vs. BM25 score) and uses only **positions**:

`RRF_score(d) = Σ 1 / (k + rank_i(d))`

In [2]:
def rrf(rankings, k=60):
    """rankings: list of ranked lists of doc ids, best first."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (k + rank)
    return sorted(scores.items(), key=lambda pair: pair[1], reverse=True)


vector_ranking = ["Section 2", "Section 7", "Section 6"]
bm25_ranking   = ["Section 6", "Section 2", "Section 7"]

for doc_id, score in rrf([vector_ranking, bm25_ranking], k=1):
    print(f"{doc_id}: {score:.3f}")

Section 2: 0.833
Section 6: 0.750
Section 7: 0.583


That reproduces the lesson's 0.833 / 0.75 / 0.583. Note that a document missing from one index's results simply gets nothing from it, so appearing in **both** lists is what lifts a document to the top.

## 3. A shared interface

`SearchIndex` is a `typing.Protocol`: any class with these two methods qualifies, with no inheritance needed. It's Python's structural version of a Java interface.

In [3]:
Document = Dict[str, Any]


class SearchIndex(Protocol):
    def add_document(self, document: Document) -> None: ...
    def search(self, query_text: str, k: int = 1) -> List[Tuple[Document, float]]: ...

### `VectorIndex`

Unlike the RAG flow notebook's version, this one embeds for you, so its API matches BM25's: it takes documents and query **text**. `add_documents` (plural) embeds a whole batch in one Voyage call, which matters with Voyage's rate limits.

In [4]:
def embed(texts, input_type):
    return voyage_client.embed(texts, model=EMBED_MODEL, input_type=input_type).embeddings


class VectorIndex:
    def __init__(self, embedding_fn=embed):
        self.embedding_fn = embedding_fn
        self.vectors, self.documents = [], []

    def add_document(self, document: Document):
        self.add_documents([document])

    def add_documents(self, documents: List[Document]):
        vectors = self.embedding_fn([d["content"] for d in documents], "document")
        self.vectors.extend(vectors)
        self.documents.extend(documents)

    @staticmethod
    def _cosine_distance(a, b):
        dot = sum(x * y for x, y in zip(a, b))
        return 1 - dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

    def search(self, query_text: str, k: int = 1):
        q = self.embedding_fn([query_text], "query")[0]
        scored = [(d, self._cosine_distance(q, v)) for v, d in zip(self.vectors, self.documents)]
        return sorted(scored, key=lambda p: p[1])[:k]          # lower distance = better

### `BM25Index`
Same scoring as the BM25 notebook, with one change for fusion: it only returns documents that match at least one query term. Otherwise a section scoring 0 would still get a rank and pick up RRF points it didn't earn.

In [5]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.documents, self.doc_tokens = [], []
        self.doc_freq = Counter()

    def add_document(self, document: Document):
        tokens = tokenize(document["content"])
        self.documents.append(document)
        self.doc_tokens.append(tokens)
        self.doc_freq.update(set(tokens))

    def add_documents(self, documents: List[Document]):
        for d in documents:
            self.add_document(d)

    def _idf(self, term):
        n, df = len(self.documents), self.doc_freq.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)

    def search(self, query_text: str, k: int = 1):
        q_terms = tokenize(query_text)
        avg_len = sum(len(t) for t in self.doc_tokens) / len(self.doc_tokens)
        scored = []
        for doc, tokens in zip(self.documents, self.doc_tokens):
            counts = Counter(tokens)
            norm = 1 - self.b + self.b * len(tokens) / avg_len
            s = sum(self._idf(t) * counts[t] * (self.k1 + 1) / (counts[t] + self.k1 * norm)
                    for t in q_terms if counts[t])
            scored.append((doc, s))
        matches = [p for p in scored if p[1] > 0]                     # non-matches earn no rank
        return sorted(matches, key=lambda p: p[1], reverse=True)[:k]  # higher score = better

## 4. The `Retriever`

A working version of the lesson's sketch. Two details the sketch leaves open:

- **Each index must return more than `k` candidates**, or RRF has nothing to fuse. `candidates` controls this.
- **Documents need a stable identity across indexes.** The same dict object is added to every index, so `id(doc)` works as the key.

`explain()` shows each document's rank in every index next to its fused score, like the lesson's table.

In [6]:
class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: Document):
        for index in self._indexes:
            index.add_document(document)

    def add_documents(self, documents: List[Document]):
        for index in self._indexes:
            if hasattr(index, "add_documents"):
                index.add_documents(documents)          # batch when supported
            else:
                for d in documents:
                    index.add_document(d)

    def _ranked(self, query_text, candidates):
        return [index.search(query_text, candidates) for index in self._indexes]

    def search(self, query_text: str, k: int = 1, k_rrf: int = 60, candidates: int = 10):
        scores, docs = defaultdict(float), {}
        for results in self._ranked(query_text, candidates):
            for rank, (doc, _) in enumerate(results, start=1):
                scores[id(doc)] += 1 / (k_rrf + rank)
                docs[id(doc)] = doc
        merged = sorted(scores.items(), key=lambda p: p[1], reverse=True)[:k]
        return [(docs[doc_id], score) for doc_id, score in merged]

    def explain(self, query_text: str, k: int = 3, k_rrf: int = 60, candidates: int = 10):
        all_results = self._ranked(query_text, candidates)
        ranks = [{id(d): r for r, (d, _) in enumerate(res, start=1)} for res in all_results]
        names = [type(i).__name__ for i in self._indexes]
        print(f"Q: {query_text}\n")
        print(f"{'section':<38}" + "".join(f"{n:>13}" for n in names) + f"{'RRF':>9}")
        for doc, score in self.search(query_text, k, k_rrf, candidates):
            row = "".join(f"{r.get(id(doc), '-'):>13}" for r in ranks)
            print(f"{doc['content'].splitlines()[0][:37]:<38}{row}{score:>9.4f}")
        print()

## 5. Build the indexes
One batched Voyage call embeds every chunk.

In [7]:
documents = [{"content": c} for c in chunks]

vector_index = VectorIndex()
bm25_index = BM25Index()
retriever = Retriever(vector_index, bm25_index)
retriever.add_documents(documents)

print(len(vector_index.documents), "docs in VectorIndex,", len(bm25_index.documents), "in BM25Index")

10 docs in VectorIndex, 10 in BM25Index


## 6. Testing the hybrid approach

The lesson's query first. The table shows each section's rank in both indexes and the fused result; a `-` means that index didn't return it in its top candidates.

In [8]:
retriever.explain("What happened with INC-2023-Q4-011?", k=3)

Q: What happened with INC-2023-Q4-011?

section                                 VectorIndex    BM25Index      RRF
## Section 5: Legal and Compliance                1            1   0.0328
## Section 4: Cybersecurity                       2            2   0.0323
## Section 1: Medical Research                    3            -   0.0159



Now the paraphrase questions where BM25 struggled last lesson, and an ID lookup where semantic search might wander. The hybrid result should hold up in both cases:

In [9]:
import time

queries = [
    "How profitable was the company?",
    "Did anything go wrong in the clinical studies?",
    "NR-204",
]

for i, q in enumerate(queries):
    attempt = 0
    while True:
        try:
            retriever.explain(q, k=3)
            break
        except Exception as exc:
            message = str(exc).lower()
            is_rate_limit = "rate" in message and ("limit" in message or "rpm" in message)
            if not is_rate_limit or attempt >= 2:
                raise
            attempt += 1
            print("Voyage rate limit reached; retrying in 60 seconds...")
            time.sleep(60)

    if i < len(queries) - 1:
        time.sleep(21)

RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

## 7. The effect of `k_rrf`

The lesson uses `k=1` for readable numbers; 60 is the common default. A small `k` makes the gap between rank 1 and rank 2 large, so one index's top pick can dominate. A large `k` flattens the curve, so agreement across indexes matters more than any single first place.

In [ ]:
for k_rrf in (1, 60):
    print(f"k_rrf = {k_rrf}")
    for rank in (1, 2, 3, 10):
        print(f"   rank {rank:>2}: 1/(k+rank) = {1 / (k_rrf + rank):.4f}")
    print()

retriever.explain("What happened with INC-2023-Q4-011?", k=3, k_rrf=1)

## 8. Extensibility: add a third index

Any class with `add_document` and `search` plugs in. This toy `TitleIndex` scores only the section header, a stand-in for a "specialized domain index". The `Retriever` code doesn't change at all.

In [ ]:
class TitleIndex:
    def __init__(self):
        self.documents = []

    def add_document(self, document: Document):
        self.documents.append(document)

    def search(self, query_text: str, k: int = 1):
        q = set(tokenize(query_text))
        scored = [(d, len(q & set(tokenize(d["content"].splitlines()[0])))) for d in self.documents]
        return sorted([p for p in scored if p[1] > 0], key=lambda p: p[1], reverse=True)[:k]


title_index = TitleIndex()
for d in documents:
    title_index.add_document(d)

retriever3 = Retriever(vector_index, bm25_index, title_index)   # reuse the already-filled indexes
retriever3.explain("What did software engineering accomplish?", k=3)

## 9. (Optional) Answer with Claude

The hybrid retriever feeds the same generation step as before.

In [ ]:
def answer_with_rag(question, k=3):
    results = retriever.search(question, k=k)
    context = "\n\n".join(
        f'<chunk index="{i}">\n{doc["content"]}\n</chunk>' for i, (doc, _) in enumerate(results)
    )
    prompt = f"""Answer the question using only the report excerpts below.
If they don't contain the answer, say so.

<report_excerpts>
{context}
</report_excerpts>

<question>{question}</question>"""
    response = anthropic_client.messages.create(
        model=MODEL, max_tokens=500, messages=[{"role": "user", "content": prompt}],
    )
    print("Retrieved:", [doc["content"].splitlines()[0] for doc, _ in results], "\n")
    print(response.content[0].text)


answer_with_rag("What happened with INC-2023-Q4-011, and did it have legal consequences?")

## Takeaways
- Scores from different indexes aren't comparable; **RRF** merges by rank, so it works across any mix of methods.
- Documents ranked well by **several** indexes rise to the top; that's how hybrid search covers both exact IDs and paraphrased questions.
- Give each index more candidates than you need, or there's nothing to fuse.
- A shared interface (`SearchIndex`) keeps indexes independent and makes adding a new retrieval method a one-line change.